In [1]:
!pip install -U transformers peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 80.3 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from huggingface_hub import login
login()

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [4]:
student_model = "google/gemma-3-1b-it"
teacher_model = "Qwen/Qwen3-8B"
device = "cuda" if torch.cuda.is_available() else "cpu" # cuda to increase the pace of learning
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

In [5]:
# the student
tokenizerGemma = AutoTokenizer.from_pretrained(student_model)
modelGemma = AutoModelForCausalLM.from_pretrained(
    student_model,
    torch_dtype=dtype, 
    device_map="auto" if torch.cuda.is_available() else None
)

# the teacher
tokenizerQwen = AutoTokenizer.from_pretrained(teacher_model)
modelQwen = AutoModelForCausalLM.from_pretrained(
    teacher_model,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None
)

if not torch.cuda.is_available():
    modelGemma = modelGemma.to(device)
    modelQwen = modelQwen.to(device)

modelGemma.eval()
modelQwen.eval()

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [6]:
print(type(modelGemma))
print(type(modelQwen))
print(modelGemma.device)
print(modelQwen.device)

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM'>
<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
cuda:0
cuda:0


In [7]:
if tokenizerGemma.pad_token is None:
    tokenizerGemma.pad_token = tokenizerGemma.eos_token

modelGemma.config.pad_token_id = tokenizerGemma.pad_token_id

In [8]:
def generate_student_answer(model, tokenizer, messages, max_new_tokens=64, **gen_kwargs):
    inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
)

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            **gen_kwargs
        )

    prompt_len = inputs['input_ids'].shape[-1]
    new_tokens = outputs[0][prompt_len:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [9]:
def generate_teacher_candidates(model, tokenizer, messages, num_candidates=8, max_new_tokens=96):
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False
    )

    inputs = tokenizer(
        [text],
        return_tensors='pt',
        padding=True,
        truncation=True
    )
    
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=50,
        num_return_sequences=num_candidates,
        pad_token_id=tokenizer.pad_token_id
    )

    prompt_len = inputs['input_ids'].shape[-1]
    answers = []

    for i in range(outputs.shape[0]):
        new_tokens = outputs[i][prompt_len:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        answers.append(answer)
    
    return answers

In [10]:
def generate_teacher_answer(model, tokenizer, messages, max_new_tokens=96):
    candidates = generate_teacher_candidates(
        model,
        tokenizer,
        messages,
        num_candidates=1,
        max_new_tokens=max_new_tokens
    )
    
    return candidates[0]

In [11]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "all")
train_dataset = dataset['train']

print(dataset)
print(train_dataset)

README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/3.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['target', 'nums', 'prompt'],
        num_rows: 80000
    })
})
Dataset({
    features: ['target', 'nums', 'prompt'],
    num_rows: 80000
})


In [12]:
example = train_dataset[0]
messages = example['prompt']
numbers = example['nums']
target = example['target']

In [13]:
teacher_answer = generate_teacher_answer(modelQwen, tokenizerQwen, messages, max_new_tokens=500)
print(teacher_answer)

To solve this problem, we need to find a way to combine the numbers 75, 80, 90, and 24 using basic arithmetic operations (+, -, *, /) so that the result is 61. Each number must be used exactly once.

Let's analyze the numbers and try different combinations.

We start by considering subtraction, since it's often useful for reducing large numbers. Let's try:

$$
90 - 80 = 10
$$

Now we have 75, 24, and 10 left. Let's try to get 61 from these.

Try:
$$
75 - 24 = 51
$$

Now we have 51 and 10. Try:
$$
51 + 10 = 61
$$

So we have:
$$
(75 - 24) + (90 - 80) = 61
$$

Let's verify:

- $75 - 24 = 51$
- $90 - 80 = 10$
- $51 + 10 = 61$

This satisfies the condition.

<answer> (75 - 24) + (90 - 80) = 61 </answer>


In [14]:
full_train = dataset['train'].shuffle(seed=42)
train_part = full_train.select(range(70000))
eval_part = full_train.select(range(70000, 80000))

# print(len(train_part), len(eval_part))
print(train_part[0].keys())

dict_keys(['target', 'nums', 'prompt'])


# Teacher checkings

In [15]:
from collections import Counter
import re

def extract_answer_block(text):
    matches = re.findall(r'<answer>\s*(.*?)\s*</answer>', text, re.DOTALL)
    if not matches:
        return None

    return matches[-1].strip()


def extract_expression(answer_block):
    if answer_block is None:
        return None

    answer_block = answer_block.strip()
    if '=' in answer_block:
        left, right = answer_block.split('=', 1)
        return left.strip()

    return answer_block


def is_allowed_expression(expression):
    if expression is None:
        return False

    return re.fullmatch(r'[0-9+\-*/().\s]+', expression) is not None


def extract_numbers_from_expression(expression):
    return [int(x) for x in re.findall(r'\d+', expression)]


def uses_numbers_correctly(expression, numbers):
    expression_numbers = extract_numbers_from_expression(expression)
    expression_counter = Counter(expression_numbers)
    numbers_counter = Counter(numbers)

    for number, count in expression_counter.items():
        if count > numbers_counter[number]:
            return False

    return True


def evaluate_expression(expression):
    if not is_allowed_expression(expression):
        return None

    try:
        value = eval(expression, {'__builtins__': None}, {})
        return value
    except Exception:
        return None


def matches_target(value, target, eps=1e-9):
    if value is None:
        return False

    return abs(value - target) < eps

In [16]:
def is_teacher_answer_correct(answer, numbers, target):
    answer_block = extract_answer_block(answer)
    if answer_block is None:
        return False, 'no <answer> block'

    expression = extract_expression(answer_block)
    if expression is None:
        return False, 'no expression'

    if not is_allowed_expression(expression):
        return False, 'forbidden symbols or operations'

    if not uses_numbers_correctly(expression, numbers):
        return False, 'numbers used incorrectly'

    value = evaluate_expression(expression)
    if value is None:
        return False, 'evaluate_expression'

    if not matches_target(value, target):
        return False, f'wrong result: got {value}, expected {target}'

    return True, 'ok'

In [17]:
is_ok, reason = is_teacher_answer_correct(teacher_answer, numbers, target)
print("Correct:", is_ok)
print("Reason:", reason)

Correct: True
Reason: ok


In [18]:
def build_task_messages(example):
    nums = example['nums']
    target = example['target']

    return [
        {
            "role": "user",
            "content": f"""Use numbers {nums} to make {target}.
Rules:
- Use only +, -, *, /
- Use each number at most once
- Do not introduce any new numbers
- Return ONLY one arithmetic expression inside <answer> </answer>
- Do NOT include an equals sign
- Do NOT include explanations
- Do NOT include <think>

Example:
<answer> (13 - 10) * 9 </answer>"""
        }
    ]

In [19]:
def build_teacher_messages(example):
    return build_task_messages(example)

In [20]:
def get_valid_teacher_answer(model, tokenizer, example, num_candidates=8, max_new_tokens=96):
    messages = build_teacher_messages(example)
    nums = example['nums']
    target = example['target']

    candidates = generate_teacher_candidates(
        model,
        tokenizer,
        messages,
        num_candidates=num_candidates,
        max_new_tokens=max_new_tokens
    )

    candidates_info = []
    for candidate in candidates:
        is_ok, reason = is_teacher_answer_correct(candidate, nums, target)
        candidates_info.append((candidate, is_ok, reason))

        if is_ok: 
            return candidate, True, 'ok', candidates_info

    return None, False, 'no valid candidate', candidates_info

In [21]:
def build_teacher_repair_messages(example, bad_answer, reason):
    nums = example['nums']
    target = example['target']

    return [
        {
            "role": "user",
            "content": f"""The previous answer was invalid.

            Numbers: {nums}
            Target: {target}
            Previous answer: {bad_answer}
            Validation error: {reason}
            
            Fix the answer.
            
            Rules:
            - Use only the provided numbers.
            - Use each provided number at most once.
            - Do not introduce new numbers.
            - Return ONLY one arithmetic expression inside <answer> </answer>.
            - Do NOT include an equals sign.
            - Do NOT include explanations.
            
            Example:
            <answer> (13 - 10) * 9 </answer>"""
        }
    ]

In [22]:
def try_teacher_repair(model, tokenizer, example, bad_answer, reason, max_new_tokens=96):
    repair_messages = build_teacher_repair_messages(example, bad_answer, reason)

    repaired_answer = generate_teacher_answer(
        model,
        tokenizer,
        repair_messages,
        max_new_tokens=max_new_tokens
    )

    nums = example['nums']
    target = example['target']

    is_ok, new_reason = is_teacher_answer_correct(repaired_answer, nums, target)
    return repaired_answer, is_ok, new_reason

In [23]:
import math

def solve_4_numbers(nums, target):
    eps = 1e-6

    def dfs(values):
        if len(values) == 1:
            value, expression = values[0]

            if abs(value - target) < eps:
                return expression
            return None

        n = len(values)
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue

                a_value, a_expression = values[i]
                b_value, b_expression = values[j]
                rest = [values[k] for k in range(n) if k != i and k != j]

                candidates = []
                candidates.append((a_value + b_value, f"({a_expression} + {b_expression})"))
                candidates.append((a_value - b_value, f"({a_expression} - {b_expression})"))
                candidates.append((a_value * b_value, f"({a_expression} * {b_expression})"))

                if abs(b_value) > eps:
                    candidates.append((a_value / b_value, f"({a_expression} / {b_expression})"))

                for new_value, new_expression in candidates:
                    result = dfs(rest + [(new_value, new_expression)])

                    if result is not None:
                        return result


        return None
    
    initial = [(float(x), str(x)) for x in nums]
    return dfs(initial)

# Getting synthetic data

In [24]:
synthetic_data = []
correct_answers = 0
total_answers = 0
# reject_reasons = Counter()
bootstrap_subset = train_part.select(range(100))

for example in bootstrap_subset:
    nums = example['nums']
    target = example['target']

    result, is_ok, reason, candidates_info = get_valid_teacher_answer(
        modelQwen,
        tokenizerQwen,
        example,
        num_candidates=3,
        max_new_tokens=96
    )

    total_answers += 1
    if is_ok:
        correct_answers += 1
        synthetic_data.append({
            'prompt': example['prompt'],
            'supervision_answer': result,
            'source': 'teacher',
            'nums': nums,
            'target': target
        })
        
        print('Accepted:', result)
        continue
        
    print('No valid candidate. Candidates:')
    for candidate, cand_ok, cand_reason in candidates_info:
        print(' ', repr(candidate), '|', cand_reason)
    #     reject_reasons[cand_reason] += 1

    solved_expression = solve_4_numbers(nums, target)
    if solved_expression is not None:
        solved_answer = f'<answer> {solved_expression} </answer>'
        correct_answers += 1
        synthetic_data.append({
            'prompt': example['prompt'],
            'supervision_answer': solved_answer,
            'source': 'solver',
            'nums': nums,
            'target': target
        })

        print('Accepted via solver:', solved_answer)
        continue

    bad_answer, _, bad_reason = candidates_info[0]
    repaired_answer, repaired_ok, repaired_reason = try_teacher_repair(
        modelQwen,
        tokenizerQwen,
        example,
        bad_answer,
        bad_reason,
        max_new_tokens=96
    )

    print('Repaired:', repr(repaired_answer))
    if repaired_ok:
        correct_answers += 1
        synthetic_data.append({
            'prompt': example['prompt'],
            'supervision_answer': repaired_answer,
            'source': 'teacher_repair',
            'nums': nums,
            'target': target
        })

        print('Accepted after repair')

    # else:
    #     reject_reasons[repaired_reason] += 1
    #     print('Repair failed:', repaired_reason)
    
teacher_accuracy = correct_answers / total_answers
print('teacher accuracy:', teacher_accuracy)
print('length of synthetic_data', len(synthetic_data))
# print('reject reasons:', reject_reasons)

No valid candidate. Candidates:
  '<answer> 43 + 38 + 14 - 19 </answer>' | wrong result: got 76, expected 89
  '<answer> (43 - 14) + (38 - 19) </answer>' | wrong result: got 48, expected 89
  '<answer> 43 + 38 + 14 - 19 </answer>' | wrong result: got 76, expected 89
Accepted via solver: <answer> (19 - (14 * (38 - 43))) </answer>
No valid candidate. Candidates:
  '<answer> (88 - 82) * (76 - 25) </answer>' | wrong result: got 306, expected 50
  '<answer> (88 - 82) * (76 - 25) </answer>' | wrong result: got 306, expected 50
  '<answer> (88 - 82) * (76 - 25) </answer>' | wrong result: got 306, expected 50
Accepted via solver: <answer> ((88 + 76) / (82 / 25)) </answer>
No valid candidate. Candidates:
  '<answer> (73 - 67) * (34 - 30) </answer>' | numbers used incorrectly
  '<answer> 73 - 67 - 34 </answer>' | wrong result: got -28, expected 40
  '<answer> (73 - 67) * (34 - 34) </answer>' | numbers used incorrectly
Accepted via solver: <answer> (73 + (34 - 67)) </answer>
No valid candidate. C

In [25]:
def build_student_messages(example):
    return build_task_messages(example)

In [26]:
def build_student_sft_example(example, tokenizer, max_length=256):
    messages = build_student_messages(example)
    answer = example['supervision_answer'].strip()

    prompt_tokens = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True
    )

    prompt_ids = prompt_tokens['input_ids']

    answer_ids = tokenizer(
        answer + tokenizer.eos_token,
        add_special_tokens=False
    )['input_ids']

    input_ids = prompt_ids + answer_ids
    input_ids = input_ids[:max_length]

    attention_mask = [1] * len(input_ids)

    prompt_len = min(len(prompt_ids), len(input_ids))
    labels = [-100] * prompt_len + input_ids[prompt_len:]

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

In [27]:
sample = build_student_sft_example(synthetic_data[0], tokenizerGemma, max_length=256)

label_ids = [x for x in sample['labels'] if x != -100]

print('Num supervised tokens:', len(label_ids))
print('Decoded supervised text:')
print(tokenizerGemma.decode(label_ids))

Num supervised tokens: 23
Decoded supervised text:
<answer> (19 - (14 * (38 - 43))) </answer><eos>


In [28]:
sample = build_student_sft_example(synthetic_data[0], tokenizerGemma, max_length=256)

print("Decoded full input:")
print(tokenizerGemma.decode(sample["input_ids"]))
print()
print("Decoded supervised target:")
label_ids = [x for x in sample["labels"] if x != -100]
print(tokenizerGemma.decode(label_ids))

Decoded full input:
<bos><start_of_turn>user
Use numbers [38, 19, 43, 14] to make 89.
Rules:
- Use only +, -, *, /
- Use each number at most once
- Do not introduce any new numbers
- Return ONLY one arithmetic expression inside <answer> </answer>
- Do NOT include an equals sign
- Do NOT include explanations
- Do NOT include <think>

Example:
<answer> (13 - 10) * 9 </answer><end_of_turn>
<start_of_turn>model
<answer> (19 - (14 * (38 - 43))) </answer><eos>

Decoded supervised target:
<answer> (19 - (14 * (38 - 43))) </answer><eos>


In [29]:
def debug_supervision_alignment(dataset_list, tokenizer, n=20):
    for i in range(min(n, len(dataset_list))):
        sample = build_student_sft_example(dataset_list[i], tokenizer, max_length=256)
        label_ids = [x for x in sample['labels'] if x != -100]
        decoded_labels = tokenizer.decode(label_ids, skip_special_tokens=False)
        expected = dataset_list[i]['supervision_answer'].strip() + tokenizer.eos_token

        print(f"\n--- Example {i} ---")
        print("EXPECTED:")
        print(repr(expected))
        print("DECODED LABELS:")
        print(repr(decoded_labels))

        if expected != decoded_labels:
            print("MISMATCH !!!")
            return

    print("\nAll checked examples look aligned.")

In [30]:
from datasets import Dataset

In [31]:
def preprocess_student(example):
    return build_student_sft_example(example, tokenizerGemma, max_length=256)

In [32]:
student_sft_dataset = Dataset.from_list(synthetic_data)
tokenized_student_dataset = student_sft_dataset.map(
    preprocess_student,
    remove_columns=student_sft_dataset.column_names
)

print(tokenized_student_dataset[0].keys())

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [33]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType 

In [34]:
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

modelGemma = get_peft_model(modelGemma, lora_config)
modelGemma.print_trainable_parameters()

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [35]:
training_args = TrainingArguments(
    output_dir='./gemma_student_sft',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    bf16=torch.cuda.is_available(),
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizerGemma, 
    model=modelGemma,
    padding=True
)

trainer = Trainer(
    model=modelGemma,
    args=training_args,
    train_dataset=tokenized_student_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
10,1.014461


TrainOutput(global_step=14, training_loss=0.8540442500795636, metrics={'train_runtime': 97.7676, 'train_samples_per_second': 2.046, 'train_steps_per_second': 0.143, 'total_flos': 115317607620096.0, 'train_loss': 0.8540442500795636, 'epoch': 2.0})

# Evaluate student

In [36]:
def evaluate_student(eval_subset, modelGemma, tokenizerGemma, max_new_tokens=96):
    correct_answers = 0
    total_answers = 0

    for example in eval_subset:
        prompt = build_student_messages(example)
        nums = example['nums']
        target = example['target']

        student_answer = generate_student_answer(
            modelGemma,
            tokenizerGemma,
            prompt,
            max_new_tokens=max_new_tokens
        )
        
        is_ok, _ = is_teacher_answer_correct(student_answer, nums, target)
        total_answers += 1

        if is_ok:
            correct_answers += 1

    task_accuracy = correct_answers / total_answers

    return {
        'task_accuracy': task_accuracy,
        'correct_answers': correct_answers,
        'total_answers': total_answers
    }

In [37]:
modelGemma.gradient_checkpointing_disable()
modelGemma.config.use_cache = True
modelGemma.eval()

train_debug = train_part.select(range(20))
print(evaluate_student(train_debug, modelGemma, tokenizerGemma, max_new_tokens=96))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'task_accuracy': 0.1, 'correct_answers': 2, 'total_answers': 20}


In [38]:
debug_subset = eval_part.select(range(5))

for i, example in enumerate(debug_subset):
    prompt = build_student_messages(example)
    answer = generate_student_answer(
        modelGemma,
        tokenizerGemma,
        prompt,
        max_new_tokens=96
    )
    
    print(f"\n--- Example {i} ---")
    print("Target:", example["target"])
    print("Nums:", example["nums"])
    print("Student output:")
    print(repr(answer))


--- Example 0 ---
Target: 27
Nums: [64, 2, 52, 17]
Student output:
'<answer> ((64 + 2) - (52 + 17)) </answer>'

--- Example 1 ---
Target: 28
Nums: [93, 33, 22, 17]
Student output:
'<answer> ((93 + 33) - (22 + 17)) </answer>'

--- Example 2 ---
Target: 71
Nums: [25, 56, 73, 29]
Student output:
'<answer> (25 + 56 - 73) </answer>'

--- Example 3 ---
Target: 85
Nums: [17, 81, 78, 10]
Student output:
'<answer> ((17 + 81) - (78 + 10)) </answer>'

--- Example 4 ---
Target: 98
Nums: [55, 2, 45]
Student output:
'<answer> (55 + 2) - (45 + 55) </answer>'


In [39]:
eval_before_gold = evaluate_student(
    eval_part.select(range(50)),
    modelGemma, 
    tokenizerGemma,
    max_new_tokens=96
)

print('before GOLD:', eval_before_gold)

before GOLD: {'task_accuracy': 0.12, 'correct_answers': 6, 'total_answers': 50}


# GOLD

In [40]:
gold_examples = []
gold_subset = train_part.select(range(100, 200))
student_correct = 0
teacher_fixed = 0
rejected = 0
total_examples = 0

for example in gold_subset:
    prompt = build_student_messages(example)
    nums = example['nums']
    target = example['target']
    total_examples += 1

    student_answer = generate_student_answer(
        modelGemma,
        tokenizerGemma,
        prompt,
        max_new_tokens=96
    )
    
    student_ok, student_reason = is_teacher_answer_correct(student_answer, nums, target)
    if student_ok:
        student_correct += 1
        gold_examples.append({
            'prompt': prompt,
            'supervision_answer': student_answer,
            'source': 'student',
            'nums': nums,
            'target': target
        })
        continue

    repaired_answer, repaired_ok, repaired_reason = try_teacher_repair(
        modelQwen,
        tokenizerQwen,
        example,
        student_answer,
        student_reason,
        max_new_tokens=96
    )

    if repaired_ok:
        teacher_fixed += 1
        gold_examples.append({
            'prompt': prompt,
            'supervision_answer': repaired_answer,
            'source': 'teacher_repair',
            'nums': nums,
            'target': target
        })
        
    else:       
        rejected += 1
            # print('Rejected example')
            # print('Student failed:', student_reason)
            # print('Teacher repair failed:', repaired_reason)

print(student_correct)
print(teacher_fixed)
print(rejected)
print(total_examples)

5
14
81
100


# Accuracy

In [41]:
task_accuracy = (student_correct + teacher_fixed) / total_examples if total_examples > 0 else 0
student_self_success_rate = student_correct / total_examples if total_examples > 0 else 0
teacher_repair_rate = teacher_fixed / total_examples if total_examples > 0 else 0
rejection_rate = rejected / total_examples if total_examples > 0 else 0

print('task_accuracy:', task_accuracy)
print('student_self_success_rate:', student_self_success_rate)
print('teacher_repair_rate:', teacher_repair_rate)
print('rejection_rate:', rejection_rate)
print('length of gold_examples:', len(gold_examples))

task_accuracy: 0.19
student_self_success_rate: 0.05
teacher_repair_rate: 0.14
rejection_rate: 0.81
length of gold_examples: 19


In [42]:
import random

student_gold = [x for x in gold_examples if x['source'] == 'student']
repair_gold = [x for x in gold_examples if x['source'] != 'student']

random.shuffle(repair_gold)
repair_gold = repair_gold[:len(student_gold) * 2] if len(student_gold) > 0 else repair_gold[:20]

gold_examples_balanced = student_gold + repair_gold
random.shuffle(gold_examples_balanced)

print("Before balancing:", Counter(x['source'] for x in gold_examples))
print("After balancing:", Counter(x['source'] for x in gold_examples_balanced))

Before balancing: Counter({'teacher_repair': 14, 'student': 5})
After balancing: Counter({'teacher_repair': 10, 'student': 5})


In [43]:
debug_supervision_alignment(synthetic_data, tokenizerGemma, n=20)
debug_supervision_alignment(gold_examples, tokenizerGemma, n=20)


--- Example 0 ---
EXPECTED:
'<answer> (19 - (14 * (38 - 43))) </answer><eos>'
DECODED LABELS:
'<answer> (19 - (14 * (38 - 43))) </answer><eos>'

--- Example 1 ---
EXPECTED:
'<answer> ((88 + 76) / (82 / 25)) </answer><eos>'
DECODED LABELS:
'<answer> ((88 + 76) / (82 / 25)) </answer><eos>'

--- Example 2 ---
EXPECTED:
'<answer> (73 + (34 - 67)) </answer><eos>'
DECODED LABELS:
'<answer> (73 + (34 - 67)) </answer><eos>'

--- Example 3 ---
EXPECTED:
'<answer> (93 - (78 - 46)) </answer><eos>'
DECODED LABELS:
'<answer> (93 - (78 - 46)) </answer><eos>'

--- Example 4 ---
EXPECTED:
'<answer> ((49 * 4) / (33 - 26)) </answer><eos>'
DECODED LABELS:
'<answer> ((49 * 4) / (33 - 26)) </answer><eos>'

--- Example 5 ---
EXPECTED:
'<answer> ((62 + 36) + (20 - 54)) </answer><eos>'
DECODED LABELS:
'<answer> ((62 + 36) + (20 - 54)) </answer><eos>'

--- Example 6 ---
EXPECTED:
'<answer> (73 - (95 - 67)) </answer><eos>'
DECODED LABELS:
'<answer> (73 - (95 - 67)) </answer><eos>'

--- Example 7 ---
EXPECTED:


# Learning student again

In [44]:
import gc

modelGemma.config.use_cache = False
modelGemma.gradient_checkpointing_enable() 

del modelQwen
gc.collect()
torch.cuda.empty_cache()

In [45]:
del trainer
gc.collect()
torch.cuda.empty_cache()

In [46]:
combined_examples = synthetic_data + gold_examples_balanced
random.shuffle(combined_examples)

combined_dataset = Dataset.from_list(combined_examples)
tokenized_combined_dataset = combined_dataset.map(
    preprocess_student,
    remove_columns=combined_dataset.column_names
)

print("synthetic:", len(synthetic_data))
print("gold:", len(gold_examples_balanced))
print("combined:", len(combined_examples))

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

synthetic: 100
gold: 15
combined: 115


In [47]:
gold_training_args = TrainingArguments(
    output_dir='./gemma_student_gold_mix',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=200,
    bf16=torch.cuda.is_available(),
    report_to='none'
)

trainer_gold = Trainer(
    model=modelGemma,
    args=gold_training_args,
    train_dataset=tokenized_combined_dataset,
    data_collator=data_collator
)

trainer_gold.train()

Step,Training Loss


TrainOutput(global_step=8, training_loss=0.3983832895755768, metrics={'train_runtime': 78.1623, 'train_samples_per_second': 1.471, 'train_steps_per_second': 0.102, 'total_flos': 66036542264064.0, 'train_loss': 0.3983832895755768, 'epoch': 1.0})

In [48]:
modelGemma.gradient_checkpointing_disable()
modelGemma.config.use_cache = True
modelGemma.eval()

train_debug = train_part.select(range(20))

debug_result = evaluate_student(
    train_debug,
    modelGemma,
    tokenizerGemma,
    max_new_tokens=96
)

print('train debug after GOLD:', debug_result)

train debug after GOLD: {'task_accuracy': 0.1, 'correct_answers': 2, 'total_answers': 20}


In [49]:
debug_subset = eval_part.select(range(5))

for i, example in enumerate(debug_subset):
    prompt = build_student_messages(example)
    answer = generate_student_answer(
        modelGemma,
        tokenizerGemma,
        prompt,
        max_new_tokens=96
    )
    print(f'\n--- Example {i} ---')
    print('Target:', example['target'])
    print('Nums:', example['nums'])
    print('Student output:')
    print(repr(answer))


--- Example 0 ---
Target: 27
Nums: [64, 2, 52, 17]
Student output:
'<answer> (52 - (17 * 2)) </answer>'

--- Example 1 ---
Target: 28
Nums: [93, 33, 22, 17]
Student output:
'<answer> (93 - (33 - 22)) </answer>'

--- Example 2 ---
Target: 71
Nums: [25, 56, 73, 29]
Student output:
'<answer> (73 - (25 + 29)) </answer>'

--- Example 3 ---
Target: 85
Nums: [17, 81, 78, 10]
Student output:
'<answer> (78 - (17 * 81)) </answer>'

--- Example 4 ---
Target: 98
Nums: [55, 2, 45]
Student output:
'<answer> (55 - (2 + 45)) </answer>'


In [50]:
eval_after_gold = evaluate_student(
    eval_part.select(range(100)),
    modelGemma,
    tokenizerGemma,
    max_new_tokens=96
)

print('after GOLD:', eval_after_gold)

after GOLD: {'task_accuracy': 0.08, 'correct_answers': 8, 'total_answers': 100}


In [52]:
def pick_best_equation_sampled(model, tokenizer, nums, target, n_samples=32):
    example = {'nums': nums, 'target': target}
    messages = build_student_messages(example)
    
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=True,
            temperature=0.7, 
            top_p=0.95,
            num_return_sequences=n_samples,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    prompt_len = inputs['input_ids'].shape[-1]
    fallback_equation = ""
    
    for i in range(outputs.shape[0]):
        new_tokens = outputs[i][prompt_len:]
        candidate_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        is_ok, _ = is_teacher_answer_correct(candidate_text, nums, target)
        
        extracted = extract_answer_block(candidate_text)
        equation = extract_expression(extracted) if extracted else candidate_text
        
        if fallback_equation == "":
            fallback_equation = equation 
            
        if is_ok:
            return True, equation 
            
    return False, fallback_equation

In [58]:
import pandas as pd
import ast
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

test_csv_path = '/kaggle/input/competitions/distillation-challenge-2026/test_public.csv'
df_test = pd.read_csv(test_csv_path)

def parse_nums(value):
    if isinstance(value, str):
        return ast.literal_eval(value) 
    return value 

if 'nums' in df_test.columns:
    df_test['nums'] = df_test['nums'].apply(parse_nums)

ds_test = df_test.to_dict('records')
submission_rows = []

modelGemma.eval()

print(f"Generating predictions for {len(ds_test)} rows...")
for index, expression in enumerate(tqdm(ds_test)):
    nums = expression["nums"]
    target = expression["target"]
    row_id = expression.get("id", index)

    is_valid, best_equation = pick_best_equation_sampled(
        modelGemma,
        tokenizerGemma,
        nums,
        target,
        n_samples=32
    )

    submission_rows.append({
        "id": row_id,
        "equation": best_equation
    })


submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)

print("\nSuccessfully saved submission.csv!")
display(submission_df.head())

Generating predictions for 2000 rows...


  0%|          | 0/2000 [00:00<?, ?it/s]


Successfully saved submission.csv!


,id,equation
0,0,(68 - (11 + 83))
1,1,(9 * (71 - 66))
2,2,51 - (82 - 51)
3,3,(97 + (59 * 52))
4,4,(74 + 98) - 20
